# Data Foundation & KPI Engineering

## Objective

This notebook transforms raw StatsBomb event data into an analysis-ready
passing dataset for evaluating football progression, performance and risk.

The workflow covers:

- match and event ingestion
- pass extraction and outcome definition
- open-play filtering
- geometric feature engineering
- spatial segmentation of the pitch
- progressive-pass classification
- preparation of datasets for Expected Threat and downstream modeling

## Analytical role

This notebook provides the data foundation for the later analysis of:

1. progression performance and Expected Threat,
2. tactical and spatial pass segments,
3. defensive context using StatsBomb 360 data, and
4. risk-adjusted decision modeling.

## 1. Data Source and Analytical Scope

### Dataset

The analysis uses the available StatsBomb Open Data sample for the
German Bundesliga 2023/24 season, covering 34 matches.

Event-level data and StatsBomb 360 contextual information are available
for this sample and are used in later stages of the project.

The raw source files are kept outside version control. This notebook loads
match metadata and event data required to construct the analytical passing dataset.

In [48]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

In [49]:
CONFIG_PATH = Path("..") / "config.json"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = json.load(f)

DATA_DIR = Path(config["statsbomb_data_dir"])

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"StatsBomb data directory not found: {DATA_DIR}"
    )

print("StatsBomb data directory loaded successfully.")

StatsBomb data directory loaded successfully.


In [50]:
COMPETITION_ID = 9
SEASON_ID = 281

### Match Scope

All available matches from the selected Bundesliga season are included.
Match metadata is loaded first so that event files can subsequently be
processed consistently across the full season.

In [51]:
matches_path = (
    DATA_DIR
    / "matches"
    / str(COMPETITION_ID)
    / f"{SEASON_ID}.json"
)

with open(matches_path, "r", encoding="utf-8") as f:
    matches = json.load(f)

matches_df = pd.DataFrame(matches)

matches_df["home_team_name"] = matches_df["home_team"].apply(
    lambda x: x["home_team_name"] if isinstance(x, dict) else None
)

matches_df["away_team_name"] = matches_df["away_team"].apply(
    lambda x: x["away_team_name"] if isinstance(x, dict) else None
)

match_scope = matches_df[
    [
        "match_id",
        "match_date",
        "home_team_name",
        "away_team_name",
        "home_score",
        "away_score",
        "match_status_360"
    ]
].copy()

print(f"Matches included: {len(match_scope)}")

match_scope.head()

Matches included: 34


,match_id,match_date,home_team_name,away_team_name,home_score,away_score,match_status_360
0,3895292,2024-04-06,Union Berlin,Bayer Leverkusen,0,1,available
1,3895320,2024-04-27,Bayer Leverkusen,VfB Stuttgart,2,2,available
2,3895158,2023-12-03,Bayer Leverkusen,Borussia Dortmund,1,1,available
3,3895107,2023-10-08,Bayer Leverkusen,FC Köln,3,0,available
4,3895340,2024-05-12,Bochum,Bayer Leverkusen,0,5,available


## 2. Pass Extraction and Outcome Definition

Event files are processed across all matches in the selected season.
Only pass events are retained and converted into a structured analytical table.

For each pass, the dataset captures:

- match and time information,
- team, passer, and recipient,
- start and end coordinates,
- play pattern and pass type,
- recorded pass outcome, and
- a binary completion indicator.

StatsBomb records an outcome only when a pass has a specific non-complete outcome.
Therefore, passes with a missing outcome are classified as completed.

In [52]:
all_pass_tables = []

for match_id in matches_df["match_id"]:
    
    events_path = DATA_DIR / "events" / f"{match_id}.json"
    
    with open(events_path, "r", encoding="utf-8") as f:
        events = json.load(f)
    
    events_df = pd.DataFrame(events)
    
    events_df["event_type"] = events_df["type"].apply(
        lambda x: x["name"] if isinstance(x, dict) else None
    )
    
    passes_df = events_df[
        events_df["event_type"] == "Pass"
    ].copy()
    
    pass_table = passes_df.copy()
    
    pass_table["match_id"] = match_id
    
    pass_table["team_name"] = pass_table["team"].apply(
        lambda x: x["name"] if isinstance(x, dict) else None
    )
    
    pass_table["player_name"] = pass_table["player"].apply(
        lambda x: x["name"] if isinstance(x, dict) else None
    )
    
    pass_table["recipient_name"] = pass_table["pass"].apply(
        lambda x: x.get("recipient", {}).get("name") if isinstance(x, dict) else None
    )
    
    pass_table["start_x"] = pass_table["location"].apply(
        lambda x: x[0] if isinstance(x, list) else None
    )
    
    pass_table["start_y"] = pass_table["location"].apply(
        lambda x: x[1] if isinstance(x, list) else None
    )
    
    pass_table["end_location"] = pass_table["pass"].apply(
        lambda x: x.get("end_location") if isinstance(x, dict) else None
    )
    
    pass_table["end_x"] = pass_table["end_location"].apply(
        lambda x: x[0] if isinstance(x, list) else None
    )
    
    pass_table["end_y"] = pass_table["end_location"].apply(
        lambda x: x[1] if isinstance(x, list) else None
    )
    
    pass_table["play_pattern_name"] = pass_table["play_pattern"].apply(
        lambda x: x["name"] if isinstance(x, dict) else None
    )
    
    pass_table["pass_type"] = pass_table["pass"].apply(
        lambda x: x.get("type", {}).get("name") if isinstance(x, dict) else None
    )
    
    pass_table["pass_outcome"] = pass_table["pass"].apply(
        lambda x: x.get("outcome", {}).get("name") if isinstance(x, dict) else None
    )
    
    pass_table["pass_success"] = pass_table["pass_outcome"].isna()
    
    pass_table = pass_table[
        [
            "match_id",
            "minute",
            "second",
            "team_name",
            "player_name",
            "recipient_name",
            "play_pattern_name",
            "pass_type",
            "pass_outcome",
            "pass_success",
            "start_x",
            "start_y",
            "end_x",
            "end_y"
        ]
    ]
    
    all_pass_tables.append(pass_table)

all_passes_df = pd.concat(all_pass_tables, ignore_index=True)

all_passes_df.head(20)

,match_id,minute,second,team_name,player_name,recipient_name,play_pattern_name,pass_type,pass_outcome,pass_success,start_x,start_y,end_x,end_y
0,3895292,0,1,Union Berlin,Brenden Aaronson,András Schäfer,From Kick Off,Kick Off,NaN,True,61.0,40.1,58.5,38.9
1,3895292,0,1,Union Berlin,András Schäfer,Danilho Doekhi,From Kick Off,NaN,NaN,True,58.8,38.6,35.9,50.9
2,3895292,0,4,Union Berlin,Danilho Doekhi,Robin Gosens,From Kick Off,NaN,NaN,True,37.9,52.2,86.1,6.0
3,3895292,0,8,Union Berlin,Robin Gosens,Yorbe Vertessen,From Kick Off,NaN,Incomplete,False,84.3,6.2,94.2,16.7
4,3895292,0,9,Bayer Leverkusen,Odilon Kossonou,Florian Wirtz,From Kick Off,Recovery,Incomplete,False,25.9,63.4,36.1,63.4
5,3895292,0,12,Bayer Leverkusen,Robert Andrich,Florian Wirtz,Regular Play,NaN,Incomplete,False,33.2,70.9,35.8,70.9
6,3895292,0,24,Bayer Leverkusen,Nathan Tella,Borja Iglesias Quintas,From Throw In,Throw-in,NaN,True,36.1,80.0,45.9,73.9
7,3895292,0,25,Bayer Leverkusen,Borja Iglesias Quintas,Nathan Tella,From Throw In,NaN,NaN,True,46.7,73.1,36.1,76.0
8,3895292,0,31,Bayer Leverkusen,Nathan Tella,Florian Wirtz,From Throw In,NaN,NaN,True,37.1,76.6,50.8,75.6
9,3895292,0,34,Union Berlin,Robin Gosens,András Schäfer,From Throw In,Recovery,Incomplete,False,70.6,6.2,76.3,9.4


In [53]:
all_passes_df.shape

(39214, 14)

## 3. Analytical Scope: Dynamic Possession vs. Set Pieces

The main analysis focuses on dynamic possession phases rather than set-piece situations.

Set-piece phases are identified using StatsBomb play patterns and separated from
the main analytical dataset because their tactical structure differs substantially
from open-play possession.

The main dataset therefore retains:

- Regular Play
- From Counter
- From Keeper
- Other

Set-piece phases are retained separately for comparison and quality checks.

In [54]:
set_piece_play_patterns = [
    "From Free Kick",
    "From Corner",
    "From Throw In",
    "From Goal Kick",
    "From Kick Off"
]

# Identify passes occurring within set-piece phases
all_passes_df["is_set_piece_phase"] = (
    all_passes_df["play_pattern_name"].isin(set_piece_play_patterns)
)

# Main analytical sample: dynamic possession
main_passes_df = all_passes_df[
    all_passes_df["play_pattern_name"].isin(
        ["Regular Play", "From Counter", "From Keeper", "Other"]
    )
].copy()

# Separate set-piece sample for comparison
set_piece_phase_passes_df = all_passes_df[
    all_passes_df["is_set_piece_phase"]
].copy()

scope_summary = pd.DataFrame({
    "dataset": ["Dynamic possession", "Set-piece phase"],
    "number_of_passes": [
        len(main_passes_df),
        len(set_piece_phase_passes_df)
    ],
    "completion_rate": [
        main_passes_df["pass_success"].mean(),
        set_piece_phase_passes_df["pass_success"].mean()
    ]
})

scope_summary["completion_rate_percent"] = (
    scope_summary["completion_rate"] * 100
).round(1)

scope_summary

,dataset,number_of_passes,completion_rate,completion_rate_percent
0,Dynamic possession,19397,0.873743,87.4
1,Set-piece phase,19817,0.826765,82.7


### Completion as an Initial Risk Indicator

Pass completion is used as a descriptive measure of execution risk.

A completed pass successfully transfers possession from the start location to the
intended end location, while an incomplete pass represents an attempted transition
that fails to preserve possession.

Completion rate therefore provides a first distinction between **potential attacking
value** and the **risk required to achieve that value**. Later stages of the project
extend this idea by combining completion probability, Expected Threat, and the
downside associated with failed passes.

## 4. Geometric Feature Engineering

Pass geometry is transformed into interpretable analytical features describing
distance, forward progression, angle, and direction.

These features are later used to distinguish simple ball circulation from
progressive actions and to support downstream statistical modeling.

In [55]:
main_passes_df["pass_length"] = np.sqrt(
    (main_passes_df["end_x"] - main_passes_df["start_x"])**2
    +
    (main_passes_df["end_y"] - main_passes_df["start_y"])**2
)

main_passes_df["forward_distance"] = (
    main_passes_df["end_x"] - main_passes_df["start_x"]
)

main_passes_df["pass_angle"] = np.degrees(
    np.arctan2(
        main_passes_df["end_y"] - main_passes_df["start_y"],
        main_passes_df["end_x"] - main_passes_df["start_x"]
    )
)

In [56]:
main_passes_df[
    [
        "pass_length",
        "forward_distance",
        "pass_angle"
    ]
].describe()

,pass_length,forward_distance,pass_angle
count,19397.000000,19397.000000,19397.000000
mean,17.926824,4.264453,-0.569879
std,11.342677,14.402303,92.081070
min,0.000000,-51.200000,-179.562636
25%,10.401923,-5.200000,-74.775570
50%,15.116878,3.500000,-1.487868
75%,22.085742,11.200000,73.651828
max,111.839036,109.100000,180.000000


In [57]:
main_passes_df["pass_direction"] = np.select(
    [
        main_passes_df["forward_distance"] > 2,
        main_passes_df["forward_distance"] < -2
    ],
    [
        "forward",
        "backward"
    ],
    default="sideways"
)

main_passes_df["pass_direction"].value_counts()

pass_direction
forward     10565
backward     6261
sideways     2571
Name: count, dtype: int64

In [58]:
direction_summary = main_passes_df.groupby("pass_direction").agg(
    number_of_passes=("pass_success", "size"),
    completion_rate=("pass_success", "mean"),
    average_pass_length=("pass_length", "mean"),
    average_forward_distance=("forward_distance", "mean")
)

direction_summary["completion_rate_percent"] = (
    direction_summary["completion_rate"] * 100
)

direction_summary

,number_of_passes,completion_rate,average_pass_length,average_forward_distance,completion_rate_percent
pass_direction,,,,,
backward,6261,0.965022,16.162059,-10.085082,96.502156
forward,10565,0.826881,20.188723,13.793800,82.688121
sideways,2571,0.844030,12.929639,0.050097,84.402956


### Initial Insight: Progression Comes With Execution Risk

Forward passes are more frequent than backward or sideways passes in the
dynamic-possession sample, but they also show a lower completion rate.

This illustrates an important trade-off:

**greater territorial progression generally requires accepting greater execution risk.**

Direction alone, however, does not measure attacking value. A forward pass may
move the ball into a low-threat area, while another pass may create a much larger
increase in scoring potential.

The next section therefore introduces spatial zones and a more meaningful
definition of progression.

## 5. Spatial Segmentation and Progressive-Pass KPI

Football space is not homogeneous: the same amount of forward movement can have
very different tactical meaning depending on where a pass starts and ends.

To make progression interpretable, the StatsBomb 120 × 80 pitch is divided into:

- defensive, middle, and attacking thirds,
- left, central, and right lanes,
- start-to-end zone transitions.

A pass is then classified as progressive if it either:

1. advances into a more advanced pitch third, or
2. moves at least 25 coordinate units forward within the same broader area.

This provides an interpretable event-data-based progression metric before
Expected Threat and defensive context are introduced in later notebooks.

In [59]:
# ---------------------------------------------------------
# Spatial zones on the StatsBomb pitch
# StatsBomb pitch dimensions:
# x-axis: 0 to 120
# y-axis: 0 to 80
# ---------------------------------------------------------

# Half of the pitch
main_passes_df["start_half"] = np.where(
    main_passes_df["start_x"] < 60,
    "own_half",
    "opponent_half"
)

main_passes_df["end_half"] = np.where(
    main_passes_df["end_x"] < 60,
    "own_half",
    "opponent_half"
)


# Thirds of the pitch
def assign_third(x):
    if x < 40:
        return "defensive_third"
    elif x < 80:
        return "middle_third"
    else:
        return "attacking_third"


# Width lanes
def assign_lane(y):
    if y < 80 / 3:
        return "left_wide"
    elif y < 2 * 80 / 3:
        return "central"
    else:
        return "right_wide"


main_passes_df["start_third"] = main_passes_df["start_x"].apply(assign_third)
main_passes_df["end_third"] = main_passes_df["end_x"].apply(assign_third)

main_passes_df["start_lane"] = main_passes_df["start_y"].apply(assign_lane)
main_passes_df["end_lane"] = main_passes_df["end_y"].apply(assign_lane)


# Full zone transition: start zone → end zone
main_passes_df["zone_transition"] = (
    main_passes_df["start_third"]
    + "_"
    + main_passes_df["start_lane"]
    + "_to_"
    + main_passes_df["end_third"]
    + "_"
    + main_passes_df["end_lane"]
)

In [60]:
pd.set_option("display.max_colwidth", None)

main_passes_df[
    [
        "start_third",
        "start_lane",
        "end_third",
        "end_lane",
        "zone_transition"
    ]
].head(10)

,start_third,start_lane,end_third,end_lane,zone_transition
5,defensive_third,right_wide,defensive_third,right_wide,defensive_third_right_wide_to_defensive_third_right_wide
20,defensive_third,central,defensive_third,left_wide,defensive_third_central_to_defensive_third_left_wide
21,defensive_third,left_wide,defensive_third,central,defensive_third_left_wide_to_defensive_third_central
22,defensive_third,central,middle_third,left_wide,defensive_third_central_to_middle_third_left_wide
23,middle_third,left_wide,middle_third,left_wide,middle_third_left_wide_to_middle_third_left_wide
24,middle_third,left_wide,middle_third,left_wide,middle_third_left_wide_to_middle_third_left_wide
25,middle_third,left_wide,attacking_third,left_wide,middle_third_left_wide_to_attacking_third_left_wide
26,defensive_third,right_wide,defensive_third,central,defensive_third_right_wide_to_defensive_third_central
27,defensive_third,central,defensive_third,left_wide,defensive_third_central_to_defensive_third_left_wide
28,middle_third,left_wide,defensive_third,left_wide,middle_third_left_wide_to_defensive_third_left_wide


In [61]:
main_passes_df["zone_transition"].value_counts().head(20)

zone_transition
middle_third_left_wide_to_middle_third_left_wide            1803
middle_third_right_wide_to_middle_third_right_wide          1776
middle_third_central_to_middle_third_central                1501
attacking_third_right_wide_to_attacking_third_right_wide    1006
attacking_third_left_wide_to_attacking_third_left_wide       870
defensive_third_central_to_defensive_third_central           742
defensive_third_left_wide_to_defensive_third_left_wide       722
middle_third_central_to_middle_third_left_wide               693
middle_third_left_wide_to_middle_third_central               675
middle_third_central_to_middle_third_right_wide              671
middle_third_right_wide_to_middle_third_central              623
defensive_third_right_wide_to_defensive_third_right_wide     589
attacking_third_central_to_attacking_third_central           466
attacking_third_right_wide_to_attacking_third_central        447
middle_third_right_wide_to_attacking_third_right_wide        443
defensive

In [62]:
third_order = {
    "defensive_third": 1,
    "middle_third": 2,
    "attacking_third": 3
}

main_passes_df["start_third_order"] = (
    main_passes_df["start_third"].astype(str).map(third_order)
)

main_passes_df["end_third_order"] = (
    main_passes_df["end_third"].astype(str).map(third_order)
)

main_passes_df["advanced_third"] = (
    main_passes_df["end_third_order"] > main_passes_df["start_third_order"]
)

In [63]:
main_passes_df["is_progressive_attempt"] = (
    (main_passes_df["advanced_third"])
    |
    (main_passes_df["forward_distance"] >= 25)
)

main_passes_df["is_completed_progressive"] = (
    main_passes_df["is_progressive_attempt"]
    &
    main_passes_df["pass_success"]
)

In [64]:
print("All main passes:", len(main_passes_df))
print("Progressive attempts:", main_passes_df["is_progressive_attempt"].sum())
print("Completed progressive passes:", main_passes_df["is_completed_progressive"].sum())

print(
    "Progressive attempt share (%):",
    main_passes_df["is_progressive_attempt"].mean() * 100
)

print(
    "Progressive completion rate (%):",
    main_passes_df.loc[
        main_passes_df["is_progressive_attempt"],
        "pass_success"
    ].mean() * 100
)

All main passes: 19397
Progressive attempts: 3342
Completed progressive passes: 2412
Progressive attempt share (%): 17.22946847450637
Progressive completion rate (%): 72.17235188509873


In [65]:
progressive_summary = main_passes_df.groupby("is_progressive_attempt").agg(
    number_of_passes=("pass_success", "size"),
    completion_rate=("pass_success", "mean"),
    average_pass_length=("pass_length", "mean"),
    average_forward_distance=("forward_distance", "mean")
)

progressive_summary["completion_rate_percent"] = (
    progressive_summary["completion_rate"] * 100
)

progressive_summary

,number_of_passes,completion_rate,average_pass_length,average_forward_distance,completion_rate_percent
is_progressive_attempt,,,,,
False,16055,0.905388,15.751562,0.335615,90.538773
True,3342,0.721724,28.376805,23.138630,72.172352


### Progression KPI: Initial Result

Progressive passes account for approximately **17.2%** of the dynamic-possession
sample.

Their completion rate is approximately **72.2%**, compared with about **90.5%**
for non-progressive passes.

Progressive actions are therefore less reliable in execution, but they also move
the ball substantially farther toward the opponent's goal. This establishes the
core **risk–progression trade-off** that is evaluated more explicitly using
Expected Threat and predictive modeling in the subsequent analysis.

In [66]:
progressive_passes_df = main_passes_df[
    main_passes_df["is_progressive_attempt"]
].copy()

progressive_passes_df["zone_transition"].value_counts().head(20)

zone_transition
middle_third_right_wide_to_attacking_third_right_wide     443
middle_third_left_wide_to_attacking_third_left_wide       392
defensive_third_left_wide_to_middle_third_left_wide       354
defensive_third_right_wide_to_middle_third_right_wide     316
defensive_third_central_to_middle_third_central           261
defensive_third_central_to_middle_third_left_wide         207
middle_third_central_to_attacking_third_central           189
defensive_third_central_to_middle_third_right_wide        183
middle_third_central_to_attacking_third_right_wide        150
middle_third_central_to_attacking_third_left_wide         145
middle_third_right_wide_to_attacking_third_central        100
defensive_third_right_wide_to_middle_third_central         90
defensive_third_left_wide_to_middle_third_central          88
middle_third_left_wide_to_attacking_third_central          73
middle_third_left_wide_to_attacking_third_right_wide       28
middle_third_right_wide_to_attacking_third_left_wide  

### Most Common Progressive Routes

The most frequent progressive transitions occur primarily through the wide channels,
especially from the middle third into the attacking third.

This indicates that wide progression is a common pattern in the sample. Frequency,
however, does not necessarily imply value. The subsequent Expected Threat analysis
therefore evaluates whether frequently used routes also generate meaningful increases
in attacking threat.

In [67]:
completed_main_passes_df = main_passes_df[
    main_passes_df["pass_success"]
].copy()

completed_progressive_passes_df = main_passes_df[
    main_passes_df["is_completed_progressive"]
].copy()

print("All main passes:", len(main_passes_df))
print("Completed main passes:", len(completed_main_passes_df))
print("Progressive attempts:", len(progressive_passes_df))
print("Completed progressive passes:", len(completed_progressive_passes_df))

All main passes: 19397
Completed main passes: 16948
Progressive attempts: 3342
Completed progressive passes: 2412


## 6. Output for Downstream Analytics

The data foundation produces two main analytical datasets:

- `main_passes_df`: all dynamic-possession pass attempts, including failed actions,
  used for completion and risk analysis.
- `completed_progressive_passes_df`: completed progressive actions, used in the
  subsequent Expected Threat analysis.

This separation preserves failed attempts for risk measurement while restricting
realized state-transition analysis to successfully completed passes.

In [68]:
PROCESSED_DIR = Path("../processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

main_passes_df.to_csv(
    PROCESSED_DIR / "main_passes_prepared.csv",
    index=False
)

completed_progressive_passes_df.to_csv(
    PROCESSED_DIR / "completed_progressive_passes_prepared.csv",
    index=False
)

print("StatsBomb data directory loaded successfully.")

StatsBomb data directory loaded successfully.
